# 03 – LSTM: training, early‑stopping su PR‑AUC, report


This notebook loads the sets prepared in **02_merge_split_scale**, trains a minimal LSTM, and evaluates on validation/test.


**Recipe**
- BCEWithLogitsLoss con `pos_weight` dal `config.json`
- Early stopping su **PR‑AUC** validation (patience=3)
- Threshold selected on **validation** (max F1), then report on **test**


In [3]:


import os, json, glob, numpy as np

BASE_RUN = "process_data/processed_lstm"
def latest_run(base=BASE_RUN):
    runs = sorted(glob.glob(os.path.join(base, "run_*")))
    assert runs, f"Nessun run trovato in {base}"
    return runs[-1]

RUN_DIR = latest_run()  # or set manually

def load_run_arrays(run_dir):
    xtr = np.load(os.path.join(run_dir, "X_train.npy")).astype(np.float32)
    ytr = np.load(os.path.join(run_dir, "y_train.npy")).astype(np.int8)
    xva = np.load(os.path.join(run_dir, "X_val.npy")).astype(np.float32)
    yva = np.load(os.path.join(run_dir, "y_val.npy")).astype(np.int8)
    xte = np.load(os.path.join(run_dir, "X_test.npy")).astype(np.float32)
    yte = np.load(os.path.join(run_dir, "y_test.npy")).astype(np.int8)

    with open(os.path.join(run_dir, "config.json"), "r", encoding="utf-8") as f:
        cfg = json.load(f)

    # per comodità/leggibilità
    features_input = cfg["input_features"]          # without label
    label_col      = cfg.get("label_col", "y_override")
    center_idx     = int(cfg.get("center_idx", xtr.shape[1]//2))
    meta_keys      = cfg.get("meta_keys", [])
    save_dir_chunks= cfg.get("save_dir_chunks", None)
    pos_w          = float(cfg.get("pos_weight_train", 1.0))

    return (xtr,ytr,xva,yva,xte,yte), cfg, {
        "features_input": features_input,
        "label_col": label_col,
        "center_idx": center_idx,
        "meta_keys": meta_keys,
        "save_dir_chunks": save_dir_chunks,
        "pos_weight_train": pos_w,
    }

(X_tr, y_tr, X_v, y_v, X_te, y_te), cfg, C = load_run_arrays(RUN_DIR)

print("RUN_DIR:", RUN_DIR)
print("X_tr:", X_tr.shape, "| X_v:", X_v.shape, "| X_te:", X_te.shape)
print("features_input:", C["features_input"])
print("label_col:", C["label_col"], "| center_idx:", C["center_idx"])


RUN_DIR: process_data/processed_lstm/run_20250826_1904
X_tr: (185156, 25, 12) | X_v: (35248, 25, 12) | X_te: (35489, 25, 12)
features_input: ['energy_Wh', 'reward_rate', 'notice_time', 'week_in_trial', 'hour_sin', 'hour_cos', 'temperature', 'wind_u', 'wind_v', 'precip_mm_log', 'ssrd_kwh', 'snr_kwh']
label_col: y_override | center_idx: 24


In [4]:
# %% 0) Setup & import
import os, glob, json, math, numpy as np, pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, precision_recall_curve
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import math
import torch
import torch.nn as nn

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Input folder: choose the last run created by notebook 02
BASE = "process_data/processed_lstm"
runs = sorted(glob.glob(os.path.join(BASE, "run_*")))
assert runs, f"Nessuna run trovata in {BASE}. Esegui prima il notebook 02."
RUN_DIR = runs[-1]
print("Using run:", RUN_DIR)


Using run: process_data/processed_lstm/run_20250826_1904


In [5]:
#  Dataset & DataLoader

class TSDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)  # (N,T,F)
        self.y = torch.from_numpy(y.astype(np.int64))  # (N,)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, idx):
        return {"x": self.X[idx], "y": self.y[idx]}

BATCH = 256
dl_tr = DataLoader(TSDataset(X_tr, y_tr), batch_size=BATCH, shuffle=True,  num_workers=0)
dl_v  = DataLoader(TSDataset(X_v,  y_v),  batch_size=BATCH, shuffle=False, num_workers=0)
dl_te = DataLoader(TSDataset(X_te, y_te), batch_size=BATCH, shuffle=False, num_workers=0)


In [7]:

# ----- Temporal attention (dot | add) ---------
class DotAttention(nn.Module):
    """
    kind='dot'  : Luong (scaled dot-product)  -> more "sharp" around t=0
    kind='add'  : Bahdanau (additive)         -> more flexible
    """
    def __init__(self, d_h, kind: str = "dot"):
        super().__init__()
        self.kind = kind
        if kind == "add":
            self.W = nn.Linear(d_h, d_h, bias=False)
            self.v = nn.Linear(d_h, 1,   bias=False)

    def forward(self, H):
        # H: (B, T, d_h)
        if self.kind == "add":
            e = self.v(torch.tanh(self.W(H))).squeeze(-1)        # (B, T)
        else:
            # "dot": usa l'ultimo stato come query (B, d_h)
            q = H[:, -1, :]                                      # (B, d_h)
            e = torch.einsum("btd,bd->bt", H, q) / math.sqrt(H.size(-1))  # (B, T)
        a = torch.softmax(e, dim=1)                              # (B, T)
        c = torch.bmm(a.unsqueeze(1), H).squeeze(1)              # (B, d_h)
        return c, a

# Alias opzionale (se vuoi chiamarla TemporalAttention altrove)
TemporalAttention = DotAttention

# ---------------------- Feature-wise gate  ----------------------
class FeatureGate(nn.Module):
    """
    Weighs channels (features) in a data-driven way
    Input X: (B, T, F) -> return Xg (B, T, F) e pesi w (B, F).
    """
    def __init__(self, in_dim, d_h=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, d_h), nn.Tanh(), nn.Linear(d_h, in_dim)
        )

    def forward(self, X):
        w  = torch.softmax(self.net(X.mean(1)), dim=-1)          # (B, F)
        Xg = X * w.unsqueeze(1)                                   # (B, T, F)
        return Xg, w

# ---------------------- LSTM + (optional) attention + (optional) gate -------
class LSTMMinimal(nn.Module):
    def __init__(self, in_dim, hidden=128,
                 use_attention=True, attn_kind="dot",
                 use_feature_gating=False):
        super().__init__()
        self.use_attention   = use_attention
        self.fgate           = FeatureGate(in_dim) if use_feature_gating else None
        self.last_feat_w     = None
        self.last_attention  = None

        self.lstm = nn.LSTM(input_size=in_dim, hidden_size=hidden, batch_first=True)

        if use_attention:
            self.attn = DotAttention(hidden, kind=attn_kind)     # oppure TemporalAttention
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):                                        # x: (B, T, F)
        if self.fgate is not None:
            x, w = self.fgate(x)
            self.last_feat_w = w
        else:
            self.last_feat_w = None

        H, _ = self.lstm(x)                                      # (B, T, h)

        if self.use_attention:
            c, a = self.attn(H)                                  # (B, h), (B, T)
            self.last_attention = a
            z = self.fc(c)                                       # (B, 1)
        else:
            z = self.fc(H[:, -1, :])                             # (B, 1)

        return z.squeeze(-1)                                     # (B,)


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = LSTMMinimal(in_dim=X_tr.shape[2],
                     hidden=128,
                     use_attention=True,       
                     attn_kind="dot",         
                     use_feature_gating=False  # True se vuoi provare la gate
                    ).to(device)


In [5]:
# train loop + export per 04

from sklearn.metrics import average_precision_score, roc_auc_score
import numpy as np, os, time

def predict_proba(model, loader):
    model.eval()
    ps, ys = [], []
    with torch.no_grad():
        for b in loader:
            xb = b["x"].to(device).float()
            yb = b["y"].numpy()
            logits = model(xb)
            p = torch.sigmoid(logits).cpu().numpy()
            ps.append(p); ys.append(yb)
    return np.concatenate(ys), np.concatenate(ps)

def train_and_eval(model, dl_tr, dl_v, epochs=30, lr=3e-4, pos_weight=1.0, patience=5):
    optim = torch.optim.AdamW(model.parameters(), lr=lr)
    lossf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))
    best, best_state, wait = -1, None, 0
    for ep in range(1, epochs+1):
        model.train()
        for b in dl_tr:
            xb = b["x"].to(device).float()
            yb = b["y"].to(device).float()
            optim.zero_grad()
            logits = model(xb)
            loss = lossf(logits, yb)
            loss.backward(); optim.step()
        # val
        y_v_arr, p_v = predict_proba(model, dl_v)
        pr = float(average_precision_score(y_v_arr, p_v))
        roc= float(roc_auc_score(y_v_arr, p_v))
        print(f"Epoch {ep:02d} | PR_AUC(val)={pr:.4f} | ROC_AUC(val)={roc:.4f}")
        # early stop su PR-AUC
        if pr > best:
            best, best_state, wait = pr, {k:v.detach().cpu() for k,v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= patience:
                print("Early stopping."); break
    if best_state is not None:
        model.load_state_dict(best_state)
    return best

# train
best_pr = train_and_eval(model, dl_tr, dl_v, epochs=40, lr=3e-4, pos_weight=C["pos_weight_train"], patience=6)

# save checkpoint
os.makedirs(os.path.join(RUN_DIR, "models"), exist_ok=True)
ckpt = os.path.join(RUN_DIR, "models", "lstm_minimal_1.pt")
torch.save(model.state_dict(), ckpt)
print("Checkpoint salvato:", ckpt)

# pred test per il 05
y_te_arr, p_te = predict_proba(model, dl_te)
np.save(os.path.join(RUN_DIR, "pred_test.npy"), p_te)

# sample attention (val) per heatmap nel 05
if getattr(model, "use_attention", False):
    model.eval()
    with torch.no_grad():
        b = next(iter(dl_v))
        x = b["x"].to(device).float()
        _ = model(x)
        alpha = getattr(model, "last_attention", None)
    if alpha is not None:
        k = min(16, alpha.shape[0])
        np.save(os.path.join(RUN_DIR, "alpha_val_sample.npy"), alpha[:k].cpu().numpy())
        np.save(os.path.join(RUN_DIR, "X_val_sample.npy"), b["x"][:k].cpu().numpy())
        np.save(os.path.join(RUN_DIR, "y_val_sample.npy"), b["y"][:k].cpu().numpy())
        print("Attention sample saved.")
    else:
        print("No attention to be saved.")


Epoch 01 | PR_AUC(val)=0.1202 | ROC_AUC(val)=0.6394
Epoch 02 | PR_AUC(val)=0.1427 | ROC_AUC(val)=0.6766
Epoch 03 | PR_AUC(val)=0.1385 | ROC_AUC(val)=0.6722
Epoch 04 | PR_AUC(val)=0.1499 | ROC_AUC(val)=0.7015
Epoch 05 | PR_AUC(val)=0.1500 | ROC_AUC(val)=0.6980
Epoch 06 | PR_AUC(val)=0.1482 | ROC_AUC(val)=0.6931
Epoch 07 | PR_AUC(val)=0.1497 | ROC_AUC(val)=0.7000
Epoch 08 | PR_AUC(val)=0.1584 | ROC_AUC(val)=0.7068
Epoch 09 | PR_AUC(val)=0.1659 | ROC_AUC(val)=0.7213
Epoch 10 | PR_AUC(val)=0.1679 | ROC_AUC(val)=0.7267
Epoch 11 | PR_AUC(val)=0.1674 | ROC_AUC(val)=0.7287
Epoch 12 | PR_AUC(val)=0.1547 | ROC_AUC(val)=0.7058
Epoch 13 | PR_AUC(val)=0.1726 | ROC_AUC(val)=0.7354
Epoch 14 | PR_AUC(val)=0.1624 | ROC_AUC(val)=0.7147
Epoch 15 | PR_AUC(val)=0.1745 | ROC_AUC(val)=0.7332
Epoch 16 | PR_AUC(val)=0.1706 | ROC_AUC(val)=0.7386
Epoch 17 | PR_AUC(val)=0.1764 | ROC_AUC(val)=0.7324
Epoch 18 | PR_AUC(val)=0.1760 | ROC_AUC(val)=0.7312
Epoch 19 | PR_AUC(val)=0.1734 | ROC_AUC(val)=0.7345
Epoch 20 | P

In [6]:
# adapt_state_dict per vecchi ckpt

def adapt_state_dict(state_raw: dict) -> dict:
    state_fix = {}
    for k,v in state_raw.items():
        nk = k
        nk = nk.replace("attn_W.", "attn.W.").replace("attn_v.", "attn.v.")
        if nk.startswith("head."): nk = nk.replace("head.", "fc.", 1)
        state_fix[nk] = v
    return state_fix

# Esempio di load legacy:
# st = torch.load("vecchio_checkpoint.pt", map_location="cpu")
# st = adapt_state_dict(st)
# model.load_state_dict(st, strict=False)


In [7]:
# %% 4) Utility metrics
def evaluate_pr_auc(model, loader):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            prob = torch.sigmoid(logits)
            ys.append(yb.cpu().numpy()); ps.append(prob.cpu().numpy())
    y = np.concatenate(ys); p = np.concatenate(ps)
    pr_auc = float(average_precision_score(y, p))
    roc = float(roc_auc_score(y, p))
    return pr_auc, roc, y, p

def tune_threshold(y, p):
    prec, rec, th = precision_recall_curve(y, p)
    th = np.append(th, 1.0)  # allinea lunghezze
    f1 = (2*prec*rec)/(prec+rec+1e-12)
    i = int(np.nanargmax(f1))
    return float(th[i]), float(f1[i])


In [8]:
for name,y in [("train",y_tr),("val",y_v),("test",y_te)]:
    print(name, "pos_rate:", float(y.mean()))


train pos_rate: 0.05810775778262654
val pos_rate: 0.06539378120744439
test pos_rate: 0.04770492265208938


In [9]:
# save attention sample 
import os, numpy as np, torch

if getattr(model, "use_attention", False):
    model.eval()
    with torch.no_grad():
        # 1) prendi un batch di validation
        batch = next(iter(dl_v))

        # 2) estrai x (e y se disponibile) sia che il batch sia dict che tupla/lista
        if isinstance(batch, dict):
            x = batch['x'].to(device)
            y = batch.get('y', None)
        elif isinstance(batch, (list, tuple)):
            x = batch[0].to(device)
            y = batch[1] if len(batch) > 1 else None
        else:
            raise TypeError(f"Batch di tipo inatteso: {type(batch)}")

        # 3) forward per aggiornare i pesi di attention nel modello
        _ = model(x)  # il forward deve impostare model.last_attention = alpha (B, T)
        alpha = getattr(model, "last_attention", None)

    if isinstance(alpha, torch.Tensor):
        k = min(16, alpha.shape[0])  # limita a 16 sequenze per non appesantire
        alpha_np = alpha[:k].detach().cpu().numpy()
        out_path = os.path.join(RUN_DIR, "alpha_val_sample.npy")
        np.save(out_path, alpha_np)
        print(f"[OK] Attention sample salvato in: {out_path} | shape={alpha_np.shape}")

        # (opzionale) salva anche input/label corrispondenti per plottare meglio nel 05
        x_np = x[:k].detach().cpu().numpy()
        np.save(os.path.join(RUN_DIR, "X_val_sample.npy"), x_np)
        if y is not None:
            y_np = y[:k].detach().cpu().numpy() if torch.is_tensor(y) else np.asarray(y[:k])
            np.save(os.path.join(RUN_DIR, "y_val_sample.npy"), y_np)
    else:
        print("Nessun attention da salvare (model.last_attention è None).")
else:
    print("Il modello non ha attention (use_attention=False).")



[OK] Attention sample salvato in: process_data/processed_lstm/run_20250826_1904/alpha_val_sample.npy | shape=(16, 25)
